# 01 — Data cleaning: FAERS GLP-1 reports
**Owner:** Rutu  ·  **Output:** the shared files everyone else uses

| Step | What happens |
|---|---|
| 0 | Setup and imports |
| 1 | Find the raw FAERS files (any quarters that are downloaded) |
| 2 | Look at one raw file to see the mess |
| 3 | Process each quarter: keep only GLP-1 primary-suspect reports |
| 4 | Remove duplicate case versions and deleted cases |
| 5 | Clean the columns (age, weight, sex, reporter, outcomes) |
| 6 | Quality checks |
| 7 | Save the shared files + pipeline counts |
| 8 | Quick summary |

All rules (drug list, serious codes, unit conversions) come from `src/definitions.py`.

## Step 0 — Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.definitions import *
from src import faers_pipeline as fp

RAW_DIR = PROJECT_ROOT / "data" / "01_raw"
OUT_DIR = PROJECT_ROOT / "data" / "02_intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
print("Project root:", PROJECT_ROOT)

## Step 1 — Find the raw files
Each FAERS quarter has 7 tables. We use 5 of them: **DEMO** (patient), **DRUG** (medicines),
**REAC** (side effects), **OUTC** (outcomes) and **INDI** (why the drug was taken).
The search works whatever folder layout the download script used.

In [ ]:
quarters = fp.find_quarter_files(RAW_DIR)
deleted_files = fp.find_deleted_files(RAW_DIR)

for q, files in quarters.items():
    missing = [t for t in fp.TABLES if t not in files]
    print(q, "->", sorted(files), "MISSING: " + str(missing) if missing else "")
print(f"\n{len(quarters)} quarters found, {len(deleted_files)} 'Deleted' file(s) found")
assert quarters, "No FAERS files found: run `python src/ingest.py` first"

## Step 2 — Look at one raw file
This shows why cleaning is needed: ages in different units (`age_cod`), weights in KG or LBS,
blank fields, and one case appearing with several `caseversion` numbers.

In [ ]:
first_q = next(iter(quarters))
demo_sample = fp.read_faers(quarters[first_q]["DEMO"])
print(first_q, "DEMO:", demo_sample.shape[0], "rows x", demo_sample.shape[1], "columns")
print("\nAge units:\n", demo_sample["age_cod"].value_counts(dropna=False).head(8))
print("\nWeight units:\n", demo_sample["wt_cod"].value_counts(dropna=False).head(5))
demo_sample.head()

## Step 3 — Process each quarter
For every quarter we:
1. Read DRUG and find rows that mention a GLP-1 drug (by ingredient or brand name).
2. Leave out insulin combination products (Xultophy, Soliqua).
3. Keep reports where the GLP-1 is the **primary suspect** (`role_cod == "PS"`).
4. Attach that report's patient info (DEMO), outcomes (OUTC), reactions (REAC) and indication (INDI).

We go **one quarter at a time** because DRUG and REAC have millions of rows each;
only the small GLP-1 part is kept in memory.

In [ ]:
reports_list, reactions_list, demo_light_list, counts_list = [], [], [], []

for q, files in quarters.items():
    rep, reac, demo_light, counts = fp.process_quarter(q, files)
    reports_list.append(rep)
    reactions_list.append(reac)
    demo_light_list.append(demo_light)
    counts_list.append(counts)
    print(f"{q}: {counts['raw_reports']:>9,} reports | GLP-1 mentioned {counts['glp1_mentioned_reports']:>7,} "
          f"| GLP-1 primary suspect {counts['glp1_ps_reports']:>7,}")

reports_all = pd.concat(reports_list, ignore_index=True)
reactions_all = pd.concat(reactions_list, ignore_index=True)
demo_light_all = pd.concat(demo_light_list, ignore_index=True)
print("\nGLP-1 primary-suspect reports before de-duplication:", len(reports_all))

## Step 4 — Remove duplicates and deleted cases
A **case** (`caseid`) can be sent several times as it is updated; each update gets a new
`caseversion` and a new `primaryid`, sometimes in a later quarter.
We keep only the **latest version** of each case, across all quarters.
FDA also lists cases that were withdrawn in a *Deleted* file; those are removed too.

In [ ]:
deleted_caseids = fp.read_deleted_caseids(deleted_files)
keep_ids, n_unique_cases = fp.latest_primaryids(demo_light_all, deleted_caseids)

mentioned_ids = set().union(*[c["glp1_mentioned_ids"] for c in counts_list])
before = len(reports_all)
reports_latest = reports_all[reports_all["primaryid"].isin(keep_ids)].copy()
reactions_latest = reactions_all[reactions_all["primaryid"].isin(keep_ids)].copy()

# Same report can appear in two quarterly files -> keep one copy (the latest quarter)
dups = reports_latest["primaryid"].duplicated().sum()
reports_latest = (reports_latest.sort_values("year_quarter")
                  .drop_duplicates("primaryid", keep="last"))
reactions_latest = reactions_latest.drop_duplicates(["primaryid", "pt"])
print("Duplicate report rows removed:", dups)

print("Deleted case ids listed by FDA:", len(deleted_caseids))
print(f"GLP-1 PS reports: {before:,} -> {len(reports_latest):,} after keeping the latest version")

## Step 5 — Clean the columns
| Column | Rule (from `src/definitions.py`) |
|---|---|
| `age_years` | Converted from YR / DEC / MON / WK / DY / HR; outside 0–110 → missing |
| `age_group` | Under 18, 18–39, 40–64, 65 and over, Unknown |
| `weight_kg` | KG kept, LBS × 0.4536; outside 30–300 → missing |
| `sex` | M, F, otherwise Unknown |
| `reporter_type`, `is_hcp` | From `occp_cod` |
| `indication_group` | Diabetes, Weight loss, Other, Unknown |
| `is_serious` | 1 if any outcome is DE, LT, HO, DS, CA or RI |

Missing values are **kept and labelled**, never dropped silently.

In [ ]:
glp1_reports = fp.clean_reports(reports_latest)

# Reaction table: add drug and target so it can be used on its own
glp1_reactions = reactions_latest.merge(
    glp1_reports[["primaryid", "drug", "is_serious"]], on="primaryid", how="inner"
)[["primaryid", "drug", "pt", "is_serious"]]

print("glp1_reports:", glp1_reports.shape)
print("glp1_reactions:", glp1_reactions.shape)
glp1_reports.head()

## Step 6 — Quality checks (every line must pass)

In [ ]:
assert glp1_reports["primaryid"].is_unique, "one row per report"
assert glp1_reports["caseid"].is_unique, "one report per case (latest version)"
assert set(glp1_reports["drug"]) <= set(GLP1_DRUGS), "only the 4 GLP-1 drugs"
assert set(glp1_reports["is_serious"]) <= {0, 1}
assert glp1_reports["age_years"].dropna().between(AGE_MIN, AGE_MAX).all()
assert glp1_reports["weight_kg"].dropna().between(WEIGHT_MIN, WEIGHT_MAX).all()
assert set(glp1_reactions["primaryid"]) <= set(glp1_reports["primaryid"])

missing_pct = (glp1_reports[["age_years", "weight_kg", "dose_amt", "route"]].isna().mean() * 100).round(1)
print("All checks passed.\n\n% missing in key columns:")
print(missing_pct.to_string())
print("Sex = Unknown:", round((glp1_reports["sex"] == "Unknown").mean() * 100, 1), "%")
print("Indication = Unknown:", round((glp1_reports["indication_group"] == "Unknown").mean() * 100, 1), "%")

## Step 7 — Save the shared files
* **Parquet** for Python (small, fast), **CSV** for Tableau.
* While only one year is downloaded, the files get that year as a suffix (`glp1_reports_2025`);
  with several years they are saved without a suffix (`glp1_reports`).
* `pipeline_counts.csv` feeds the funnel chart C1.

In [ ]:
years = sorted({q[:4] for q in quarters})
suffix = f"_{years[0]}" if len(years) == 1 else ""

glp1_reports.to_parquet(OUT_DIR / f"glp1_reports{suffix}.parquet", index=False)
glp1_reports.to_csv(OUT_DIR / f"glp1_reports{suffix}.csv", index=False)
glp1_reactions.to_parquet(OUT_DIR / f"glp1_reactions{suffix}.parquet", index=False)
glp1_reactions.to_csv(OUT_DIR / f"glp1_reactions{suffix}.csv", index=False)

pipeline_counts = pd.DataFrame([
    {"step": "1. Raw reports (all drugs)", "reports": sum(c["raw_reports"] for c in counts_list)},
    {"step": "2. Unique cases (latest version, deleted removed)", "reports": n_unique_cases},
    {"step": "3. Mention a GLP-1 drug", "reports": len(mentioned_ids & keep_ids)},
    {"step": "4. GLP-1 is the primary suspect (final)", "reports": len(glp1_reports)},
])
pipeline_counts.to_csv(OUT_DIR / f"pipeline_counts{suffix}.csv", index=False)

for f in sorted(OUT_DIR.glob(f"*{suffix}.*")):
    print(f"{f.name:40s} {f.stat().st_size / 1e6:8.2f} MB")
pipeline_counts

## Step 8 — Quick summary (sanity check, not a slide chart)

In [ ]:
summary = (glp1_reports.groupby("drug")
           .agg(reports=("primaryid", "count"),
                pct_serious=("is_serious", "mean"),
                median_age=("age_years", "median"),
                pct_female=("sex", lambda s: (s == "F").mean()))
           .sort_values("reports", ascending=False))
summary["pct_serious"] = (summary["pct_serious"] * 100).round(1)
summary["pct_female"] = (summary["pct_female"] * 100).round(1)
print("Overall % serious:", round(glp1_reports["is_serious"].mean() * 100, 1))
summary